# 03b — Harmonização LULC

Harmonização dos produtos COS, SIOSE e SIOSE AR através da tabela `lookup_lulc_harmonizacao.csv`.

O notebook lê um ou vários GeoPackages, recorta diretamente à área de estudo, aplica a nomenclatura harmonizada e guarda apenas o produto final. 

## Importações

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

sys.path.append("/code/scripts")

from lulc_utils import harmonize_lulc, read_lulc

## Configuração

In [ ]:
# ALTERAR APENAS ESTA CÉLULA

# Para COS: area = "centro"
# Para SIOSE e SIOSE AR: area = "extremadura"
area = "extremadura"

# Opções: "cos", "siose", "siose_ar"
source = "siose_ar"

# COS:      1995, 2007, 2010, 2015, 2018
# SIOSE:    2005, 2009, 2011, 2014
# SIOSE AR: 2017, 2020
year = 2017

In [ ]:
base = f"/code/data/processed/{area}"

cos_folder = "/code/data/raw/lulc"
siose_folder = "/code/pen/data/SIOSE"

aoi_file = f"{base}/aoi/{area}.shp"

# A tabela é comum aos produtos portugueses e espanhóis.
harm_file = (
    f"/code/data/processed/{area}/lulc/"
    "harmonized_tables/lookup_lulc_harmonizacao.csv"
)

out_harm = f"{base}/lulc/harmonized_vectors"
Path(out_harm).mkdir(parents=True, exist_ok=True)

## Fontes de dados

In [ ]:
cos = {
    1995: {
        "files": [f"{cos_folder}/COS1995v2-S1.gpkg"],
        "layers": ["COS1995v2"],
        "n1": "COS95n1_C",
        "code": "COS95n4_C",
        "label": "COS95n4_L",
    },
    2007: {
        "files": [f"{cos_folder}/COS2007v3-S1.gpkg"],
        "layers": ["COS2007v3"],
        "n1": "COS07n1_C",
        "code": "COS07n4_C",
        "label": "COS07n4_L",
    },
    2010: {
        "files": [f"{cos_folder}/COS2010v2-S1.gpkg"],
        "layers": ["COS2010v2"],
        "n1": "COS10n1_C",
        "code": "COS10n4_C",
        "label": "COS10n4_L",
    },
    2015: {
        "files": [f"{cos_folder}/COS2015v2-S1.gpkg"],
        "layers": ["COS2015v2"],
        "n1": "COS15n1_C",
        "code": "COS15n4_C",
        "label": "COS15n4_L",
    },
    2018: {
        "files": [f"{cos_folder}/COS2018v2-S1.gpkg"],
        "layers": ["COS2018v2"],
        "n1": "COS18n1_C",
        "code": "COS18n4_C",
        "label": "COS18n4_L",
    },
}

siose = {
    2005: {
        "files": [
            f"{siose_folder}/SIOSE_Extremadura_2005_GPKG/"
            "SIOSE_Extremadura_2005.gpkg"
        ],
        "layers": ["T_POLIGONOS"],
        "code": "CODIIGE",
        "label": None,
    },
    2009: {
        "files": [
            f"{siose_folder}/SIOSE_Extremadura_2009_GPKG/"
            "SIOSE_Extremadura_2009.gpkg"
        ],
        "layers": ["T_POLIGONOS"],
        "code": "CODIIGE",
        "label": None,
    },
    2011: {
        "files": [
            f"{siose_folder}/SIOSE_Extremadura_2011_GPKG/"
            "SIOSE_Extremadura_2011.gpkg"
        ],
        "layers": ["T_POLIGONOS"],
        "code": "CODIIGE",
        "label": None,
    },
    2014: {
        "files": [
            f"{siose_folder}/SIOSE_Extremadura_2014_GPKG/"
            "SIOSE_Extremadura_2014.gpkg"
        ],
        "layers": ["T_POLIGONOS"],
        "code": "CODIIGE",
        "label": None,
    },
}

siose_ar = {
    2017: {
        "files": [
            f"{cos_folder}/SIOSE_2017_Badajoz.gpkg",
            f"{cos_folder}/SIOSE_2017_Caceres.gpkg",
        ],
        # A layer é identificada automaticamente após a extração.
        "layers": [
            "t_poligonos",
            "t_poligonos",
        ],
        "code": "ID_COBERTURA_MAX",
        "label": None,
    },
    2020: {
        "files": [
            f"{siose_folder}/SIOSE_2020_Badajoz.gpkg",
            f"{siose_folder}/SIOSE2020_Caceres.gpkg",
        ],
        # A layer é identificada automaticamente após a extração.
        "layers": [
            "T_POLIGONOS",
            "T_POLIGONOS",
        ],
        "code": "ID_COBERTURA_MAX",
        "label": None,
    },
}

sources = {
    "cos": cos,
    "siose": siose,
    "siose_ar": siose_ar,
}

## Produto selecionado

In [ ]:
if source not in sources:
    raise ValueError(f"Fonte inválida: {source}")

if year not in sources[source]:
    raise ValueError(f"Ano {year} não disponível para {source}.")

cfg = sources[source][year]

product = {
    "cos": "cos_1995" if year == 1995 else "cos_2007_2018",
    "siose": "siose",
    "siose_ar": "siose_ar",
}[source]

name = f"{source}_{year}"
harm_out = f"{out_harm}/{name}_harm.gpkg"
harm_layer = f"{name}_harm"
drop_report = f"{out_harm}/{name}_classes_excluidas.csv"

print("Produto:", source)
print("Ano:", year)
print("Ficheiros de origem:", len(cfg["files"]))
print("Output:", harm_out)

## Leitura da área de estudo e da tabela de harmonização

In [ ]:
import sqlite3


files = [
    "/code/data/raw/lulc/SIOSE_2017_Badajoz.gpkg",
    "/code/data/raw/lulc/SIOSE_2017_Caceres.gpkg",
]


for file in files:
    print("\nFicheiro:", file)

    try:
        with sqlite3.connect(
            f"file:{file}?mode=ro",
            uri=True,
        ) as conn:
            result = conn.execute(
                "PRAGMA quick_check;"
            ).fetchall()

        print("Resultado:", result)

    except Exception as error:
        print("ERRO:", error)

In [ ]:
aoi = gpd.read_file(aoi_file)

harm = pd.read_csv(
    harm_file,
    dtype={
        "product": str,
        "code_original": str,
        "label_harmonizada": str,
        "incluir_modelo": int,
        "obs": str,
        "id_harm": "Int64",
    },
)

print("AOI:", aoi_file)
print("Produto na tabela:", product)

## Leitura e recorte do produto LULC

In [ ]:
where = None

if source == "cos":
    # Excluir territórios artificializados e massas de água.
    where = (
        f'CAST("{cfg["n1"]}" AS TEXT) '
        "NOT IN ('1', '9')"
    )

lulc_clip = read_lulc(
    files=cfg["files"],
    layers=cfg["layers"],
    aoi=aoi,
    where=where,
)

print("Feições após o recorte:", len(lulc_clip))

## Harmonização

In [ ]:
lulc_harm, classes_excluidas, summary = harmonize_lulc(
    lulc=lulc_clip,
    lookup=harm,
    product=product,
    code_field=cfg["code"],
    label_field=cfg["label"],
)

classes_excluidas.to_csv(drop_report, index=False)

display(classes_excluidas.head(20))
print("Classes sem correspondência:", len(classes_excluidas))

## Exportação

In [ ]:
Path(harm_out).unlink(missing_ok=True)

lulc_harm.to_file(
    harm_out,
    layer=harm_layer,
    driver="GPKG",
)

print("Vetor harmonizado:", harm_out)
print("Relatório:", drop_report)

## Verificação final

In [ ]:
classes = (
    lulc_harm[["id_harm", "classe_harm"]]
    .value_counts()
    .rename("n")
    .reset_index()
    .sort_values(["id_harm", "classe_harm"])
)

display(classes)

execution = pd.DataFrame([{
    "fonte": source,
    "ano": year,
    "ficheiros_lidos": len(cfg["files"]),
    **summary,
    "output": harm_out,
}])

display(execution)